In [1]:
import opendatasets as od

# This triggers a Kaggle login prompt (username + API key) the first time.
# On Windows, if prompted for a "kaggle.json" file location, it typically expects
# it at C:\Users\<YourUsername>\.kaggle\kaggle.json — download this API key file
# from your Kaggle account settings page ("Create New API Token") beforehand.
dataset_url = 'https://www.kaggle.com/c/new-york-city-taxi-fare-prediction/overview'
od.download(dataset_url)

data_dir = './new-york-city-taxi-fare-prediction'

Skipping, found downloaded files in ".\new-york-city-taxi-fare-prediction" (use force=True to force download)


In [2]:
import pandas as pd

# Only read the columns we actually need, with the SMALLEST dtype that safely fits each column.
# float32 instead of the pandas default float64 halves memory for coordinate/fare columns.
# uint8 for passenger_count since it never exceeds ~9.
dtypes = {
    'fare_amount': 'float32',
    'pickup_longitude': 'float32',
    'pickup_latitude': 'float32',
    'dropoff_longitude': 'float32',
    'dropoff_latitude': 'float32',
    'passenger_count': 'uint8'
}

# Set our maximum known limit, the percentage to load from disk, and the percentage to randomly sample.
# Calculating these dynamically lets us easily scale the data volume up or down.
max_rows = 5_000_000
read_percent = 0.1
sample_percent = 0.1

# nrows caps how many rows pandas even parses from disk — this alone is the biggest speed win.
# parse_dates converts pickup_datetime straight into a proper datetime64 column instead of a string.
df = pd.read_csv(
    f'{data_dir}/train.csv',
    nrows=int(max_rows * read_percent),
    dtype=dtypes,
    parse_dates=['pickup_datetime']
)

# Case study explicitly suggests working with ~1% of data for iteration speed.
# random_state=42 is set for reproducibility — same sample every run, so results are comparable across experiments.
df = df.sample(frac=sample_percent, random_state=42)  # 10% of the 500k pre-filtered rows ≈ 50k rows, a workable size
print(df.shape)

(50000, 8)


In [3]:
# Function to handle a single API request
def fetch_prediction(index, row):
    dt = row['pickup_datetime']
    
    payload = {
        "pickup_lat": float(row['pickup_latitude']),
        "pickup_lon": float(row['pickup_longitude']),
        "dropoff_lat": float(row['dropoff_latitude']),
        "dropoff_lon": float(row['dropoff_longitude']),
        "hour": int(dt.hour),
        "day_of_week_num": int(dt.dayofweek),
        "month": int(dt.month),
        "passenger_count": int(row['passenger_count'])
    }
    
    try:
        response = requests.post(API_URL, json=payload, headers=HEADERS)
        response.raise_for_status() 
        
        prediction_data = response.json()
        predicted_fare = prediction_data.get('fare_amount')
        actual_fare = row['fare_amount']
        difference = predicted_fare - actual_fare
        
        # Return the successful result as a dictionary
        return {
            'index': index,
            'actual': actual_fare,
            'predicted': predicted_fare,
            'difference': difference,
            'error': None
        }
    except requests.exceptions.RequestException as e:
        # Return the error if it fails
        return {'index': index, 'error': str(e)}

In [ ]:
import requests
import concurrent.futures

# Ensure pickup_datetime is a datetime object so we can extract time components
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

API_URL = 'http://localhost:5000/predict'
HEADERS = {'Content-Type': 'application/json'}

# 1. GRAB EXACTLY 20 RANDOM ROWS
n_samples = min(100, len(df))
sample_df = df.sample(n=n_samples, random_state=42)

# List to store our comparison results
comparisons = []

print(f"{'Index':<8} | {'Actual Fare':<12} | {'Predicted':<12} | {'Difference':<12}")
print("-" * 52)

# 2. RUN API REQUESTS IN PARALLEL
# max_workers=10 means it will process up to 10 requests simultaneously
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    # Submit all tasks to the thread pool
    futures = [
        executor.submit(fetch_prediction, index, row) 
        for index, row in sample_df.iterrows()
    ]
    
    # Process results as they complete (order may vary based on response time)
    for future in concurrent.futures.as_completed(futures):
        result = future.result()
        
        if result['error']:
            print(f"Row {result['index']} failed API call: {result['error']}")
        else:
            print(f"{result['index']:<8} | ${result['actual']:<11.2f} | ${result['predicted']:<11.2f} | ${result['difference']:<11.2f}")
            comparisons.append(result)

# 3. CALCULATE METRICS
if comparisons:
    results_df = pd.DataFrame(comparisons)
    mae = results_df['difference'].abs().mean()
    print("-" * 52)
    print(f"Mean Absolute Error (MAE) across {len(results_df)} predictions: ${mae:.2f}")

Index    | Actual Fare  | Predicted    | Difference  
----------------------------------------------------
227725   | $8.00        | $8.49        | $0.49       
432158   | $6.10        | $5.83        | $-0.27      
456821   | $8.10        | $8.83        | $0.73       
249719   | $20.50       | $17.62       | $-2.88      
77617    | $14.00       | $12.48       | $-1.52      
98333    | $28.10       | $28.25       | $0.15       
88357    | $7.30        | $19.35       | $12.05      
356573   | $4.00        | $5.06        | $1.06       
202922   | $13.30       | $19.32       | $6.02       
154900   | $13.70       | $15.27       | $1.57       
457942   | $19.70       | $16.58       | $-3.12      
79057    | $6.00        | $4.99        | $-1.01      
106374   | $7.30        | $7.25        | $-0.05      
210456   | $34.04       | $21.98       | $-12.06     
228148   | $34.90       | $32.19       | $-2.71      
456916   | $6.50        | $8.52        | $2.02       
172410   | $14.50       | $16